### 0. Data Preparation

In [ ]:
import notebook
print(notebook.__version__)

import sys
print(sys.version)
!pip list

In [ ]:
# ==========================================
# 0. DATA PREPARATION & GLOBAL CONFIGURATION
# ==========================================
import MDAnalysis as mda
import MDAnalysis.transformations as trans
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from scipy.spatial import Voronoi, ConvexHull
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
import warnings
warnings.filterwarnings('ignore')

# ---------------------------------------------------------
# FONT SIZE CONFIGURATION (Change these to adjust all plots)
# ---------------------------------------------------------
# INDICATOR: Modify these integers to instantly rescale text across all steps.
F_TITLE = 16     # Font size for plot titles
F_LABEL = 18     # Font size for X and Y axis labels
F_TICK = 18      # Font size for axis tick numbers
F_LEGEND = 18    # Font size for legends
# ---------------------------------------------------------

# 0.1 Data Loading using MDAnalysis
print("Phase 0.1: Loading system...")

topology_file = 'file_name.pdb'
trajectory_file = 'file_name.dcd'

u = mda.Universe(topology_file, trajectory_file)

protein = u.select_atoms('all')

# **********************************************************
# Setting system parameters
number_of_molecules = 200 
atoms_per_molecule = len(protein) // number_of_molecules
stride = 1 

box_dims = np.array([1000.0, 1000.0, 1000.0, 90.0, 90.0, 90.0])
box_volume = 1000.0 * 1000.0 * 1000.0
# **********************************************************

# 0.2 Molecule Reconstruction and Periodic Boundary Condition Handling
print("Phase 0.2: Applying PBC handling and Condensate Reconstruction...")
u.atoms.guess_bonds() 

from MDAnalysis.transformations.boxdimensions import set_dimensions
inject_box = set_dimensions(box_dims)
make_whole = trans.unwrap(protein)
pin_to_centre = trans.center_in_box(protein, center='mass')

u.trajectory.add_transformations(inject_box, make_whole, pin_to_centre)

# 0.3 Simplifying Complex Data: Centre of Geometry (CoG) extraction
print("Phase 0.3: Extracting Centre of Geometry (CoG) for all frames...")
all_cogs = [] 

for ts in u.trajectory[::stride]:
    current_cogs = np.array([
        u.atoms[i * atoms_per_molecule : (i + 1) * atoms_per_molecule].center_of_geometry()
        for i in range(number_of_molecules)
    ])
    all_cogs.append(current_cogs)

all_cogs = np.array(all_cogs)
final_cogs = all_cogs[-1]

# --- THE TIMELINE METADATA FIX ---
# Manually map the 101 frames evenly across 2-microsecond (2000 ns) production timeline.
time_steps = np.linspace(0, 2000, len(all_cogs))
# ----------------------------------

print("Phase 0 Complete. System ready for analysis.")

### 1. Single-chain conformational state

In [ ]:
# ==========================================
# 1. SINGLE-CHAIN CONFORMATIONAL STATE
# ==========================================
print("Phase 1: Analysing intramolecular conformations...")
from MDAnalysis.analysis import align, rms, distances

single_chain = u.atoms[0 : atoms_per_molecule]

rg_list = []
ree_list = []
persistence_lengths = []

# 1.1 Rg, 1.2 Ree, and 1.5 Persistence Length Tracking
for frame_idx in range(0, len(u.trajectory), stride):
    ts = u.trajectory[frame_idx]
    ts.dimensions = box_dims
    
    rg_list.append(single_chain.radius_of_gyration())
    ree_list.append(np.linalg.norm(single_chain.positions[0] - single_chain.positions[-1]))
    
    # 1.5 Coarse-grained Persistence Length calculation using bond vector correlations
    positions = single_chain.positions
    vectors = np.diff(positions, axis=0)
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    
    # Calculate the actual physical average bond length in Angstroms
    # This prevents the assumption that beads are exactly 1.0 A apart
    avg_bond_length = np.mean(norms)
    unit_vectors = vectors / (norms + 1e-8)
    
    cosines = []
    for i in range(len(unit_vectors) - 1):
        cosines.append(np.dot(unit_vectors[i], unit_vectors[i+1]))
    
    mean_cos = np.mean(cosines) if len(cosines) > 0 else 0
    if mean_cos > 0 and mean_cos < 1:
        # Multiply the logarithmic decay by the physical bond length
        # This converts the metric from 'beads' to physical Angstroms (A)
        lp = avg_bond_length * (-1.0 / np.log(mean_cos))
    else:
        lp = 0.0
    persistence_lengths.append(lp)

# 1.4 Root Mean Square Fluctuation (RMSF)
aligner = align.AlignTraj(u, u, select=f"bynum 1:{atoms_per_molecule}", in_memory=True).run()
rmsf_analysis = rms.RMSF(single_chain).run()
rmsf_data = rmsf_analysis.results.rmsf

# 1.3 Intramolecular Contact Map & 1.6 Secondary Structure Proxy (Final Frame)
ts = u.trajectory[-1]
ts.dimensions = box_dims
intra_dist = distances.distance_array(single_chain.positions, single_chain.positions)

# 1.6 Secondary Structure Local Compaction Window (Moving 5-bead Rg window)
local_compaction = []
window_size = 5
for idx in range(atoms_per_molecule - window_size + 1):
    fragment = single_chain[idx : idx + window_size]
    local_compaction.append(fragment.radius_of_gyration())

# Individual Plot 1.1 & 1.2: Chain Compaction Over Time
plt.figure(figsize=(8, 5))
plt.plot(time_steps, rg_list, label='Rg', color='blue')
plt.plot(time_steps, ree_list, label='Ree', color='red', alpha=0.7)
plt.title("1.1 & 1.2: Chain Compaction Over Time", fontsize=F_TITLE)
plt.xlabel("Time (ns)", fontsize=F_LABEL)
plt.ylabel("Distance (Å)", fontsize=F_LABEL)
plt.tick_params(axis='both', labelsize=F_TICK)
plt.legend(fontsize=F_LEGEND)
plt.grid(True, linestyle=':', alpha=0.5)
plt.ylim(0, 110)
plt.tight_layout()
plt.show()

# Individual Plot 1.3: Intramolecular Contact Map
plt.figure(figsize=(8, 6))
plt.imshow(intra_dist, cmap='viridis', aspect='auto', vmin=0, vmax=60)
plt.title("1.3: Intramolecular Contact Map", fontsize=F_TITLE)
plt.xlabel("Residue Index", fontsize=F_LABEL)
plt.ylabel("Residue Index", fontsize=F_LABEL)
plt.tick_params(axis='both', labelsize=F_TICK)
cbar = plt.colorbar()
cbar.set_label("Distance (Å)", fontsize=F_LABEL)
cbar.ax.tick_params(labelsize=F_TICK)
plt.tight_layout()
plt.show()

# Individual Plot 1.4: Root Mean Square Fluctuation
plt.figure(figsize=(8, 5))
plt.plot(range(atoms_per_molecule), rmsf_data, color='purple')
plt.title("1.4: Root Mean Square Fluctuation", fontsize=F_TITLE)
plt.xlabel("Residue Index", fontsize=F_LABEL)
plt.ylabel("RMSF (Å)", fontsize=F_LABEL)
plt.tick_params(axis='both', labelsize=F_TICK)
plt.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()

# Individual Plot 1.5: Structural Persistence Length
plt.figure(figsize=(8, 5))
plt.plot(time_steps, persistence_lengths, color='crimson', linewidth=2)
plt.title("1.5: Structural Persistence Length (l_p)", fontsize=F_TITLE)
plt.xlabel("Time (ns)", fontsize=F_LABEL)
plt.ylabel("Persistence Length (Å)", fontsize=F_LABEL)
plt.tick_params(axis='both', labelsize=F_TICK)
plt.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()

# Individual Plot 1.6: Secondary Structure Compaction Profiling
plt.figure(figsize=(8, 5))
plt.plot(range(len(local_compaction)), local_compaction, color='olive', linewidth=2)
plt.title("1.6: Secondary Structure Compaction Profiling", fontsize=F_TITLE)
plt.xlabel("Chain Profile Position", fontsize=F_LABEL)
plt.ylabel("Local 5-Bead Rg (Å)", fontsize=F_LABEL)
plt.tick_params(axis='both', labelsize=F_TICK)
plt.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()

# ==========================================================
# 1.7 SECONDARY STRUCTURE ANALYSIS (DSSP via MDTraj)
# ==========================================================
print("Phase 1.7: Calculating Alpha-Helicity using MDTraj DSSP...")
import mdtraj as md

# MDTraj requires reloading the trajectory using its own engine.
# Please ensure 'topology_file' and 'trajectory_file' variables match your file paths.
try:
    # Load trajectory into MDTraj
    traj_md = md.load(trajectory_file, top=topology_file)
    
    # Run DSSP (Dictionary of Secondary Structure of Proteins)
    # simplified=True returns 'H' (Helix), 'E' (Strand), 'C' (Coil)
    dssp_results = md.compute_dssp(traj_md, simplified=True)
    
    # Calculate the percentage of residues in Alpha-Helix ('H') conformation per frame
    # dssp_results shape is (n_frames, n_residues)
    helix_fraction = np.mean(dssp_results == 'H', axis=1) * 100.0
    
    # ==========================================
    # VISUALISATION PLOT FOR SECTION 1.7
    # ==========================================
    plt.figure(figsize=(8, 5))
    plt.plot(time_steps, helix_fraction, color='darkorange', linewidth=2.5, label='% Alpha-Helix')
    
    # Overlay the overall mean helicity
    mean_helicity = np.mean(helix_fraction)
    plt.axhline(mean_helicity, color='black', linestyle='--', linewidth=2, 
                label=f'Mean Helicity: {mean_helicity:.1f}%')
    
    plt.title("1.7: Alpha-Helicity Evolution over Time (HyRes DSSP)", fontsize=F_TITLE)
    plt.xlabel("Time (ns)", fontsize=F_LABEL)
    plt.ylabel("Alpha-Helix Content (%)", fontsize=F_LABEL)
    plt.tick_params(axis='both', labelsize=F_TICK)
    
    # Set y-axis range to compare two plots
    plt.ylim(-2, 105) 
    
    plt.legend(fontsize=F_LEGEND)
    plt.grid(True, linestyle=':', alpha=0.5)
    plt.tight_layout()
    plt.show()

    print(f"DSSP Analysis Complete. Average Helicity: {mean_helicity:.2f}%")

except Exception as e:
    print(f"Error loading files with MDTraj: {e}")
    print("Please ensure you have defined 'topology_file' and 'trajectory_file' as strings.")

### 2. Intermolecular organisation / phase structure

In [ ]:
# ==========================================================
# 2. INTERMOLECULAR ORGANISATION / PHASE STRUCTURE
# ==========================================================
print("Phase 2: Analysing phase structure and density...")
from MDAnalysis.analysis import rdf

# 2.1 Radial Distribution Function (EPS Calibration)
cog_universe = mda.Universe.empty(
    n_atoms=number_of_molecules,
    n_residues=number_of_molecules,
    atom_resindex=np.arange(number_of_molecules),
    n_segments=number_of_molecules,
    trajectory=True
)

cog_universe.add_TopologyAttr('resname', ['MOL'] * number_of_molecules)
cog_universe.add_TopologyAttr('name', ['COG'] * number_of_molecules)
cog_universe.load_new(all_cogs, format='Memory')

for ts in cog_universe.trajectory:
    ts.dimensions = np.array([1000.0, 1000.0, 1000.0, 90.0, 90.0, 90.0])

cog_group = cog_universe.select_atoms("all")
# Set range to 500.0 Å to properly capture your 112.50 Å EPS peak
rdf_analyzer = rdf.InterRDF(cog_group, cog_group, nbins=100, range=(0.0, 500.0))
rdf_analyzer.run()

g_r = rdf_analyzer.results.rdf
rdf_bins = rdf_analyzer.results.bins
first_peak_idx = np.argmax(g_r)
first_min_idx = first_peak_idx + np.argmin(g_r[first_peak_idx : first_peak_idx + 20])
optimum_eps = rdf_bins[first_min_idx]


# 2.2 DBSCAN Clustering
# First, run the standard clustering using the accurate RDF derived cutoff
db = DBSCAN(eps=optimum_eps, min_samples=3).fit(final_cogs)
final_labels = db.labels_

# Automated Sensitivity Analysis (Robustness Check)
# Calculate the +/- 10% thresholds
eps_minus_10 = optimum_eps * 0.90
eps_plus_10 = optimum_eps * 1.10

# Run hidden DBSCANs to test the physical boundaries
labels_minus = DBSCAN(eps=eps_minus_10, min_samples=3).fit(final_cogs).labels_
labels_plus = DBSCAN(eps=eps_plus_10, min_samples=3).fit(final_cogs).labels_

# Count the number of distinct clusters formed in each scenario
clusters_main = len(np.unique(final_labels[final_labels != -1]))
clusters_minus = len(np.unique(labels_minus[labels_minus != -1]))
clusters_plus = len(np.unique(labels_plus[labels_plus != -1]))

# Print the diagnostic report
print("\n=== PHASE 2.3 SENSITIVITY ANALYSIS ===")
print(f"Optimal EPS ({optimum_eps:.2f} Å)   : {clusters_main} clusters found")
print(f"-10% EPS    ({eps_minus_10:.2f} Å)   : {clusters_minus} clusters found")
print(f"+10% EPS    ({eps_plus_10:.2f} Å)   : {clusters_plus} clusters found")

if clusters_minus == clusters_main == clusters_plus:
    print("Status: ROBUST. The condensate morphology is stable against minor epsilon fluctuations.")
else:
    print("Status: SENSITIVE. Minor changes in epsilon alter the cluster count. Interpret results with caution.")
print("======================================\n")


# 2.3 Cluster Size Distribution (Phase Morphology)
unique, counts = np.unique(final_labels[final_labels != -1], return_counts=True)

# 2.4 Voronoi Tessellation
dummy_points = np.array([[2000,2000,2000], [-1000,-1000,-1000], [2000,-1000,-1000], [-1000,2000,2000]])
voronoi_points = np.vstack([final_cogs, dummy_points])
vor = Voronoi(voronoi_points)

volumes = []
expected_avg_vol = box_volume / number_of_molecules
for region_index in vor.point_region[:number_of_molecules]:
    region = vor.regions[region_index]
    if -1 not in region and len(region) > 0:
        hull = ConvexHull(vor.vertices[region])
        volumes.append(hull.volume)
valid_volumes = [v for v in volumes if v < (expected_avg_vol * 3)]

# ==========================================
# SEPARATED INDIVIDUAL VISUALISATION PLOTS
# ==========================================

# Plot 2.1: Radial Distribution Function
plt.figure(figsize=(8, 5))
plt.plot(rdf_bins, g_r, color='black', linewidth=2)
plt.axvline(optimum_eps, color='red', linestyle='--', label=f'RDF Min EPS: {optimum_eps:.1f} Å')
plt.title("2.1: Radial Distribution Function", fontsize=F_TITLE)
plt.xlabel("Distance (Å)", fontsize=F_LABEL)
plt.ylabel("g(r)", fontsize=F_LABEL)
plt.tick_params(axis='both', labelsize=F_TICK)
plt.legend(fontsize=F_LEGEND)
plt.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()

# Plot 2.3: Cluster Size Distribution
plt.figure(figsize=(8, 5))
plt.bar(unique if len(unique) > 0 else [0], counts if len(counts) > 0 else [0], color='green')
plt.title("2.3: Cluster Size Distribution", fontsize=F_TITLE)
plt.xlabel("Cluster ID", fontsize=F_LABEL)
plt.ylabel("Number of Proteins", fontsize=F_LABEL)
plt.tick_params(axis='both', labelsize=F_TICK)
plt.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()

# Plot 2.4: Voronoi Cell Volume Distribution
plt.figure(figsize=(8, 5))
plt.hist(valid_volumes, bins=30, color='darkorange', edgecolor='black', alpha=0.7)
plt.title("2.4: Voronoi Cell Volume Distribution", fontsize=F_TITLE)
plt.xlabel("Local Volume per Protein (Å³)", fontsize=F_LABEL)
plt.ylabel("Frequency", fontsize=F_LABEL)
plt.tick_params(axis='both', labelsize=F_TICK)
plt.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()

# Plot 2.5: 3D Spatial Distribution of Condensates
# Creating a new 3D spatial plot to visualise the clustered droplets and dilute phase
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Colouring the beads based on their assigned DBSCAN cluster ID (-1 is unclustered/dilute)
scatter = ax.scatter(
    final_cogs[:, 0], 
    final_cogs[:, 1], 
    final_cogs[:, 2], 
    c=final_labels, 
    cmap='plasma', 
    s=100, 
    edgecolors='k', 
    alpha=0.8,
    vmin=-1,
    vmax=10
)

ax.set_title("2.5: 3D Spatial Distribution of Condensates", fontsize=F_TITLE)
ax.set_xlabel("X (Å)", fontsize=F_LABEL)
ax.set_ylabel("Y (Å)", fontsize=F_LABEL)
ax.set_zlabel("Z (Å)", fontsize=F_LABEL)
ax.tick_params(axis='both', labelsize=F_TICK)

# Adding a colour bar to distinguish specific clusters from the background
cbar = plt.colorbar(scatter, ax=ax, pad=0.1)
cbar.set_label("Cluster ID (-1 is Dilute Phase)", fontsize=F_LABEL)
cbar.ax.tick_params(labelsize=F_TICK)

plt.tight_layout()
plt.show()

### 3. Connectivity / network formation

In [ ]:
# ==========================================
# 3. CONNECTIVITY & NETWORK FORMATION
# ==========================================
print("Phase 3: Mapping intermolecular network and specific chemistry...")
from MDAnalysis.analysis import distances

# 3.1 Network Graph Theory
G = nx.Graph()
for i in range(number_of_molecules):
    G.add_node(i)
for i in range(len(final_cogs)):
    for j in range(i + 1, len(final_cogs)):
        if np.linalg.norm(final_cogs[i] - final_cogs[j]) <= optimum_eps:
            G.add_edge(i, j)

# 3.2 Specific Contacts (Heatmap Matrix)
unique_residues = list(set(u.atoms.resnames))
# Print all unique residue names present in your system topology
print("Unique residue names in the system:")
print(list(set(u.atoms.resnames)))
# Add to ignore list if there are SOL, WAT, HOH, TIP3 / TIP4 / PW, W, W4, POL
ignore_list = ['W', 'ION', 'NA', 'CL', 'AMN', 'CBX', 'SOL', 'WAT', 'HOH', 'TIP3', 'TIP4', 'PW', 'W', 'W4', 'POL'] 
aa_types = [res for res in unique_residues if res not in ignore_list]

num_aa = len(aa_types)
interaction_heatmap = np.zeros((num_aa, num_aa))
contact_cutoff = 8.0 

for mol_i in range(number_of_molecules):
    atoms_i = u.atoms[mol_i * atoms_per_molecule : (mol_i + 1) * atoms_per_molecule]
    for mol_j in range(mol_i + 1, number_of_molecules):
        atoms_j = u.atoms[mol_j * atoms_per_molecule : (mol_j + 1) * atoms_per_molecule]
        for i, resA in enumerate(aa_types):
            for j, resB in enumerate(aa_types):
                if j >= i: 
                    groupA = atoms_i.select_atoms(f"resname {resA}")
                    groupB = atoms_j.select_atoms(f"resname {resB}")
                    if len(groupA) > 0 and len(groupB) > 0:
                        dist_mat = distances.distance_array(groupA.positions, groupB.positions)
                        contacts = np.count_nonzero(dist_mat <= contact_cutoff)
                        interaction_heatmap[i, j] += contacts
                        if i != j:
                            interaction_heatmap[j, i] += contacts 

# Visualisations for Phase 3
fig = plt.figure(figsize=(14, 6))

# Plot 3.1: 2D Intermolecular Network Projection
plt.figure(figsize=(8, 6))
pos = {i: (final_cogs[i, 0], final_cogs[i, 1]) for i in range(number_of_molecules)}
nx.draw_networkx_nodes(G, pos, node_size=30, node_color='mediumpurple', edgecolors='black')
nx.draw_networkx_edges(G, pos, alpha=0.3, edge_color='dimgray')
plt.title("3.1: 2D Intermolecular Network Projection", fontsize=F_TITLE)
plt.tick_params(axis='both', labelsize=F_TICK)
plt.axis('on')
plt.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()

# Plot 3.2: Specific Intermolecular Contacts Heatmap
plt.figure(figsize=(9, 7))
# Set vmax to fixed number to compare two plots
plt.imshow(interaction_heatmap, cmap='Reds', aspect='auto', vmin=0, vmax=25000)
plt.xticks(np.arange(num_aa), aa_types, rotation=45, fontsize=F_TICK)
plt.yticks(np.arange(num_aa), aa_types, fontsize=F_TICK)
plt.title("3.2: Specific Intermolecular Contacts", fontsize=F_TITLE)
plt.xlabel("Amino Acid Type", fontsize=F_LABEL)
plt.ylabel("Amino Acid Type", fontsize=F_LABEL)

for i in range(num_aa):
    for j in range(num_aa):
        text_color = "white" if interaction_heatmap[i, j] > (np.max(interaction_heatmap) / 2) else "black"
        plt.text(j, i, int(interaction_heatmap[i, j]), ha="center", va="center", color=text_color, fontsize=8)

plt.colorbar(label="Total Physical Contacts")
plt.tight_layout()
plt.show()
# ==========================================================
# 3.3 CONTACT LIFETIME ANALYSIS
# ==========================================================
print("Phase 3.3: Analysing intermolecular contact lifetimes...")
from MDAnalysis.analysis import distances

# Computational scaling: sample the first 20 molecules to track 
# continuous lifetimes against all other molecules in the system.
sample_size = min(20, number_of_molecules)
active_bonds = {}       # Maps (global_atom_i, global_atom_j) -> consecutive_frames_bound
recorded_lifetimes = []

# Extract time per step from your existing time_steps array
dt = (time_steps[1] - time_steps[0]) if len(time_steps) > 1 else 1.0

# Track bond durations sequentially through the entire trajectory
for frame_idx in range(0, len(u.trajectory), stride):
    ts = u.trajectory[frame_idx]
    ts.dimensions = box_dims
    
    # Store all physical contacts present in the current frame
    current_frame_contacts = set()
    
    for m_i in range(sample_size):
        atoms_i = u.atoms[m_i * atoms_per_molecule : (m_i + 1) * atoms_per_molecule]
        for m_j in range(m_i + 1, number_of_molecules):
            atoms_j = u.atoms[m_j * atoms_per_molecule : (m_j + 1) * atoms_per_molecule]
            
            # Compute distance matrix for this specific molecular pair
            d_mat = distances.distance_array(atoms_i.positions, atoms_j.positions)
            row_indices, col_indices = np.where(d_mat <= contact_cutoff)
            
            for r, c in zip(row_indices, col_indices):
                global_i = m_i * atoms_per_molecule + r
                global_j = m_j * atoms_per_molecule + c
                current_frame_contacts.add((global_i, global_j))
                
    # Evaluate lifetime transitions between frames
    # 1. Update existing contacts that are still sustained
    for bond_pair in list(active_bonds.keys()):
        if bond_pair in current_frame_contacts:
            active_bonds[bond_pair] += 1
        else:
            # The bond has broken; log its total lifetime in nanoseconds
            recorded_lifetimes.append(active_bonds[bond_pair] * stride * dt)
            del active_bonds[bond_pair]
            
    # 2. Register completely new contacts formed in this frame
    for bond_pair in current_frame_contacts:
        if bond_pair not in active_bonds:
            active_bonds[bond_pair] = 1

# Append any remaining active bonds still bound at the final frame
for bond_pair, frame_duration in active_bonds.items():
    recorded_lifetimes.append(frame_duration * stride * dt)

# ==========================================
# VISUALISATION PLOT FOR SECTION 3.3
# ==========================================
plt.figure(figsize=(8, 5))

if len(recorded_lifetimes) > 0:
    # Plot log-scaled histogram to cleanly capture long-lived sticker vs short-lived spacer behaviors
    plt.hist(recorded_lifetimes, bins=30, color='crimson', edgecolor='black', alpha=0.7, log=True)
    
    # Calculate and overlay mean lifetime property
    mean_lifetime = np.mean(recorded_lifetimes)
    plt.axvline(mean_lifetime, color='black', linestyle='--', linewidth=2, 
                label=f'Mean Lifetime: {mean_lifetime:.2f} ns')
    plt.legend(fontsize=F_LEGEND) # INDICATOR: Configured globally
else:
    plt.text(0.5, 0.5, "No stable intermolecular contacts recorded.", ha='center', va='center')

plt.title("3.3: Intermolecular Contact Lifetime Distribution", fontsize=F_TITLE) # INDICATOR
plt.xlabel("Contact Duration (ns)", fontsize=F_LABEL) # INDICATOR
plt.ylabel("Frequency (Log Scale)", fontsize=F_LABEL) # INDICATOR
plt.tick_params(axis='both', labelsize=F_TICK) # INDICATOR
plt.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()

print(f"Analysis complete. Logged {len(recorded_lifetimes)} distinct contact events.")

### 4. Dynamics / material properties

In [ ]:
# ==========================================
# 4. DYNAMICS & MATERIAL PROPERTIES
# ==========================================
print("Phase 4: Analysing dynamics and material tracking...")
import MDAnalysis.analysis.msd as msd

# 4.1 Mean Square Displacement
# NOTE: Full translational drift correction is complete.
# The trans.center_in_box operation executed on the trajectory structure 
# during Phase 0.2 handles this by holding the whole matrix centered.
if len(u.trajectory) > 1:
    msd_analyzer = msd.EinsteinMSD(u, select='all', msd_type='xyz', fft=True)
    msd_analyzer.run()
    msd_values = msd_analyzer.results.timeseries 
    msd_times = np.arange(len(msd_values))

    # Visualisation for Phase 4.1
    plt.figure(figsize=(8, 5))
    plt.plot(msd_times, msd_values, color='forestgreen', linewidth=2)
    plt.title("4.1: Mean Square Displacement", fontsize=F_TITLE) # INDICATOR
    plt.xlabel("Time Lag (frames)", fontsize=F_LABEL) # INDICATOR
    plt.ylabel("MSD (Å²)", fontsize=F_LABEL) # INDICATOR
    #  Set y-axis range to compare two plots
    # plt.ylim(0, 850000)
    plt.tick_params(axis='both', labelsize=F_TICK) # INDICATOR
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.show()
else:
    print("Skipping MSD: Trajectory contains an individual single frame configuration.")

# ==========================================================
# 4.2 RESIDENCE TIME ANALYSIS
# ==========================================================
print("Phase 4.2: Analysing molecule residence times inside clusters...")

n_frames = all_cogs.shape[0]
dt = (time_steps[1] - time_steps[0]) if len(time_steps) > 1 else 1.0

# Track cluster history for each molecule
# Dictionary maps: molecule_id -> consecutive frames spent in a cluster
mol_cluster_frames = {i: 0 for i in range(number_of_molecules)}
residence_times = []

for f_idx in range(n_frames):
    cogs_frame = all_cogs[f_idx]
    
    # Run DBSCAN on the current frame coordinates to track cluster states over time
    db_frame = DBSCAN(eps=optimum_eps, min_samples=3).fit(cogs_frame)
    labels_frame = db_frame.labels_
    
    for mol_id in range(number_of_molecules):
        if labels_frame[mol_id] != -1:
            # Molecule is currently localized inside a cluster
            mol_cluster_frames[mol_id] += 1
        else:
            # Molecule has escaped to the dilute phase; log duration if it just left
            if mol_cluster_frames[mol_id] > 0:
                residence_times.append(mol_cluster_frames[mol_id] * dt)
                mol_cluster_frames[mol_id] = 0

# Flush remaining active tracks at the final frame boundary
for mol_id, frames_bound in mol_cluster_frames.items():
    if frames_bound > 0:
        residence_times.append(frames_bound * dt)

# ==========================================
# VISUALISATION PLOT FOR SECTION 4.2
# ==========================================
plt.figure(figsize=(8, 5))
if len(residence_times) > 0:
    plt.hist(residence_times, bins=30, color='royalblue', edgecolor='black', alpha=0.7)
    mean_res = np.mean(residence_times)
    plt.axvline(mean_res, color='black', linestyle='--', linewidth=2, 
                label=f'Mean Residence: {mean_res:.2f} ns')
    plt.legend(fontsize=F_LEGEND) # INDICATOR: Configured globally
else:
    plt.text(0.5, 0.5, "No cluster residence events recorded.", ha='center', va='center')

plt.title("4.2: Molecular Residence Time inside Condensates", fontsize=F_TITLE) # INDICATOR
plt.xlabel("Residence Duration (ns)", fontsize=F_LABEL) # INDICATOR
plt.ylabel("Frequency", fontsize=F_LABEL) # INDICATOR
plt.tick_params(axis='both', labelsize=F_TICK) # INDICATOR
plt.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()

# ==========================================================
# 4.3 BOND SURVIVAL PROBABILITY
# ==========================================================
print("Phase 4.3: Analysing intermolecular bond survival probability...")

n_frames = all_cogs.shape[0]
dt = (time_steps[1] - time_steps[0]) if len(time_steps) > 1 else 1.0

# Define maximum time lag to track (evaluating up to 100 steps into the future)
max_lag = min(100, n_frames // 2)
time_lags = np.arange(max_lag) * dt

# Sample frame origins (t0) across your trajectory to gather clean statistics
step_t0 = max(1, n_frames // 50)
t0_frames = range(0, n_frames - max_lag, step_t0)

total_initial_bonds = 0
surviving_bonds_count = np.zeros(max_lag)

# Helper function to extract a set of active molecular pair bonds in a specific frame
def get_bonds_set(frame_idx):
    cogs = all_cogs[frame_idx]
    bonds = set()
    for i in range(number_of_molecules):
        for j in range(i + 1, number_of_molecules):
            if np.linalg.norm(cogs[i] - cogs[j]) <= optimum_eps:
                bonds.add((i, j))
    return bonds

# Calculate autocorrelation over the sampled time origins
for t0 in t0_frames:
    initial_bonds = get_bonds_set(t0)
    n_bonds_t0 = len(initial_bonds)
    
    if n_bonds_t0 == 0:
        continue
        
    total_initial_bonds += n_bonds_t0
    
    for lag in range(max_lag):
        current_bonds = get_bonds_set(t0 + lag)
        # Determine how many of the original molecular bonds remain continuously present
        still_alive = len(initial_bonds.intersection(current_bonds))
        surviving_bonds_count[lag] += still_alive

# Normalise counts to generate a clean probability decay curve
if total_initial_bonds > 0:
    bond_survival_prob = surviving_bonds_count / (total_initial_bonds / len(t0_frames))
    bond_survival_prob = bond_survival_prob / bond_survival_prob[0] # Normalise start to 1.0
else:
    bond_survival_prob = np.zeros(max_lag)

# ==========================================
# VISUALISATION PLOT FOR SECTION 4.3
# ==========================================
plt.figure(figsize=(8, 5))
plt.plot(time_lags, bond_survival_prob, color='darkviolet', linewidth=2.5, label='S(τ)')
plt.title("4.3: Intermolecular Bond Survival Probability", fontsize=F_TITLE) # INDICATOR
plt.xlabel("Time Lag τ (ns)", fontsize=F_LABEL) # INDICATOR
plt.ylabel("Survival Probability S(τ)", fontsize=F_LABEL) # INDICATOR
plt.tick_params(axis='both', labelsize=F_TICK) # INDICATOR
plt.ylim(-0.05, 1.05)
plt.grid(True, linestyle=':', alpha=0.5)
plt.legend(fontsize=F_LEGEND) # INDICATOR
plt.tight_layout()
plt.show()